# Panorama Stitching Pipeline (Corners → ANMS → Descriptors → Matching → RANSAC → Warp/Blend)

# Introduction

This notebook implements an end-to-end panorama stitching pipeline using classical computer vision. The pipeline:

- Detects interest points (Harris or Shi–Tomasi)
- Applies Adaptive Non-Maximal Suppression (ANMS) to retain a spatially well-distributed subset
- Builds simple patch-based descriptors (40×40 → Gaussian blur → 8×8 downsample → normalize)
- Matches descriptors with a ratio test on SSD distances
- Estimates a robust homography via RANSAC
- Warps images into a common frame and blends the overlap

All code is designed to run locally without Colab-specific steps.

# System Overview

Pipeline steps executed in this notebook:
1) Corner detection
2) Adaptive Non-Maximal Suppression (ANMS)
3) Patch-based feature descriptors (normalized 8×8)
4) Descriptor matching with a ratio test
5) RANSAC-based homography estimation
6) Image warping and blending into a panorama

The following sections contain concise explanations and the corresponding implementation for each step.

Local setup and data

- Provide your own overlapping image sequences, or use any public samples.
- Place images under a local data directory for this project. Example structure:
  - projects/panorama-stitching/data/TestSet1/*.jpg
  - projects/panorama-stitching/data/TestSet2/*.jpg
  - projects/panorama-stitching/data/TestSet3/*.jpg
- By default, the notebook reads from ../data and writes outputs to ../results (relative to this notebook).
- You can override these with environment variables PANORAMA_DATA_DIR and PANORAMA_RESULTS_DIR.

In [ ]:
# Setup (paths and imports)
import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Paths (repository-relative by default)
DATA_DIR = os.environ.get('PANORAMA_DATA_DIR', '../data')
RESULTS_DIR = os.environ.get('PANORAMA_RESULTS_DIR', '../results')
os.makedirs(RESULTS_DIR, exist_ok=True)

print('DATA_DIR =', DATA_DIR)
print('RESULTS_DIR =', RESULTS_DIR)

In [ ]:
# Quick data sanity check
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np

train_image = mpimg.imread(f"{DATA_DIR}/TestSet1/1.jpg")
plt.imshow(train_image)
plt.axis("off")
plt.show()

## ANMS output

The cells below visualize corner detection and ANMS-selected keypoints on example images.

#### Step 1: Corner Detection

We detect interest points using a classical corner detector (Harris or Shi–Tomasi). In this notebook, Harris corner response is computed and a fraction of the strongest responses are retained as candidate keypoints. cv2.goodFeaturesToTrack is an interchangeable alternative.

In [ ]:
### Corner Detection
# Convert to grayscale and compute Harris response; threshold to keep strong corners.

import cv2


def detect_corner(img):

    # convert to gray scale
    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Harris response and simple thresholding
    corners = []
    dst = cv2.cornerHarris(img, 2, 3, 0.04)
    largest = dst.max()
    for i in range(dst.shape[0]):
        for j in range(dst.shape[1]):
            if dst[i, j] > 0.01 * largest:
                corners.append((j, i))

    return np.array(corners), dst

In [ ]:
# Visualize detected corners on an example image

img = cv2.imread(f"{DATA_DIR}/TestSet2/1.jpg")
corners, _ = detect_corner(img)

for x, y in corners:

    cv2.circle(img, (x, y), 3, (0, 0, 255), -1)

plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

#### Step 2: Adaptive Non-Maximal Suppression (ANMS)

Raw corner responses are often clustered in textured regions. ANMS retains a spatially well-distributed subset by assigning each candidate a suppression radius: the squared distance to the nearest stronger corner. Sorting by this radius and keeping the top N selects points that are both strong and well separated.

In [ ]:
### Adaptive Non-Maximal Suppression (or ANMS)
# Perform ANMS: Adaptive Non-Maximal Suppression
# Show ANMS output as an image


# ASSUMPTIONS:
# cmap is output from cv2.cornerHarris
# ANMS is implemented completely seperate from part 1
def ANMS(cmap, corners, num_best):

    coordinates = corners

    radius = np.full(shape=(coordinates.shape[0], 1), fill_value=999999999999999)

    coordinates = coordinates.T
    Xarray = coordinates[1]
    Yarray = coordinates[0]

    for i in range(len(Xarray)):
        for j in range(len(Xarray)):

            if cmap[Xarray[j], Yarray[j]] > cmap[Xarray[i], Yarray[i]]:

                ED = (Xarray[j] - Xarray[i]) ** 2 + (Yarray[j] - Yarray[i]) ** 2
                if ED < radius[i]:
                    radius[i] = ED

    # get sorted indicies
    sorted_indices = np.argsort(-radius, axis=0).reshape((1, -1))[0]

    out = np.zeros(shape=(num_best, 2), dtype=int)

    # grab NStrongest
    for i in range(0, num_best):
        out[i] = (Xarray[sorted_indices[i]], Yarray[sorted_indices[i]])

    return out

In [ ]:
# Visualize corner density vs. ANMS selection

# no ANMS
img = cv2.imread(f"{DATA_DIR}/TestSet2/1.jpg")
corners, _ = detect_corner(img)
for x, y in corners:
    cv2.circle(img, (x, y), 3, (0, 0, 255), -1)
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()


# w/ ANMS
img = cv2.imread(f"{DATA_DIR}/TestSet2/1.jpg")
corners, cmap = detect_corner(img)
corners = ANMS(cmap, corners, 30)
for x, y in corners:
    cv2.circle(img, (y, x), 3, (0, 0, 255), -1)
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

### 2. Feature Descriptors

In this step we build a simple, illumination-robust descriptor per keypoint:

- Extract a 40×40 patch centered at the keypoint (or shift slightly if too close to the border).
- Apply a small Gaussian blur.
- Downsample to 8×8 and reshape to a 64×1 vector.
- Standardize to zero mean and unit variance.

This compact descriptor works well with SSD distance and a ratio test.

In [ ]:
# NOTE: this version off centers any corners near the edge of the image
def feature_descript(gray_img, corners):

    res = []
    for x, y in corners:

        start_row = x - 20
        end_row = x + 20
        if start_row < 0:
            start_row += -start_row
            end_row += -start_row
        if end_row > gray_img.shape[0] - 1:
            end_row -= end_row - gray_img.shape[0] - 1
            start_row -= end_row - gray_img.shape[0] - 1

        start_col = y - 20
        end_col = y + 20
        if start_col < 0:
            start_col += -start_col
            end_col += -start_col
        if end_col > gray_img.shape[1] - 1:
            end_col -= end_col - gray_img.shape[1] - 1
            start_col -= end_col - gray_img.shape[1] - 1

        patch = gray_img[start_row:end_row, start_col:end_col]
        blurred_patch = cv2.GaussianBlur(patch, (3, 3), 0)
        sub_sampled = cv2.resize(blurred_patch, (8, 8), interpolation=cv2.INTER_AREA)

        reshaped = sub_sampled.astype(int).reshape((64, -1))
        final = (reshaped - np.mean(reshaped)) / np.std(reshaped)
        res.append((final, (x, y)))

    return res

In [ ]:
# Inspect ANMS-selected corners and example 8×8 descriptors

# show corners we are aiming for
img = cv2.imread(f"{DATA_DIR}/TestSet2/1.jpg")
corners, cmap = detect_corner(img)
corners = ANMS(cmap, corners, 30)
for x, y in corners:
    cv2.circle(img, (y, x), 3, (0, 0, 255), -1)
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

# show feature descriptors
img = cv2.imread(f"{DATA_DIR}/TestSet2/1.jpg")
descriptors = feature_descript(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), corners)
for item, _ in descriptors:
    plt.imshow(item.reshape((8, 8)), cmap="gray")
    plt.axis("off")
    plt.show()

### 3. Feature Matching

Given 64-D descriptors per keypoint, we match features across a pair of images by computing SSD distances. For each descriptor in image 1, we sort candidates in image 2 and apply a ratio test between the best and second-best distances; matches passing the threshold are retained. We visualize correspondences using OpenCV drawMatches.

## Feature Matching

Feature correspondences are visualized using OpenCV drawMatches in the next cell.

In [ ]:
### Feature matching

# Features1 is the output from calling feature_descript() on the corners in img1
# so it consits of (featureVector, location)
# where location is a tuple of form (x,y)


# high level overview:
# find differences[i][j]
# for each corner[i] in image1 only match it w/ sortedSSDs[i][j] if there is a significant difference between sortedSSDs[i][j] and sortedSSDs[i][j+1]
# this makes sure we only grab correspondences that have a high degree of certainty
def feature_match(img1, img2, features1, features2):

    # differences[i][j] = difference between features1[i] and features2[j]
    differences = np.zeros((len(features1), len(features2)))

    # compute SSD for all points
    # for each feature in img1
    for i, (featureVector1, _) in enumerate(features1):
        # for each feature in img2
        for j, (featureVector2, _) in enumerate(features2):
            # for the 64 pixels in each feature
            for k in range(64):
                differences[i, j] += (featureVector1[k][0] - featureVector2[k][0]) ** 2

    # list of cv2.DMatch objects
    dmatches = []

    ratio = 0.66
    # find good matches
    for i in range(differences.shape[0]):
        # obtain argsort of differences[i]
        argsort = np.argsort(differences[i])

        # for all other differences conditionally append them for matches for i
        for index in range(len(argsort) - 1):
            if (
                differences[i, argsort[index]] / differences[i, argsort[index + 1]]
                < ratio
            ):
                dmatches.append(
                    cv2.DMatch(
                        _queryIdx=i,
                        _trainIdx=argsort[index],
                        _distance=differences[i, argsort[index]],
                    )
                )

            else:
                break

    return dmatches

In [ ]:
# Compute and visualize tentative feature matches between two images

# get feature descriptors for corners in image 1
img1 = cv2.imread(f"{DATA_DIR}/TestSet2/2.jpg")
initial_corners1, cmap1 = detect_corner(img1)
corners1 = ANMS(cmap1, initial_corners1, 100)
features1 = feature_descript(cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY), corners1)

# get feature descriptors for corners in image 2
img2 = cv2.imread(f"{DATA_DIR}/TestSet2/3.jpg")
initial_corners2, cmap2 = detect_corner(img2)
corners2 = ANMS(cmap2, initial_corners2, 100)
features2 = feature_descript(cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY), corners2)

# run feature_match
matches = feature_match(-1, -1, features1, features2)
print(f"len(matches) {len(matches)}")

# convert corners into key point objects
kp1 = []
for x, y in corners1:
    kp1.append(cv2.KeyPoint(y.astype(float), x.astype(float), 1))

# convert corners into key point objects
kp2 = []
for x, y in corners2:
    kp2.append(cv2.KeyPoint(y.astype(float), x.astype(float), 1))

# run drawMatches to create the final image
out = cv2.drawMatches(
    cv2.cvtColor(img1, cv2.COLOR_BGR2RGB),
    kp1,
    cv2.cvtColor(img2, cv2.COLOR_BGR2RGB),
    kp2,
    matches,
    None,
)

plt.figure(figsize=(16, 12))
plt.imshow(out)
plt.axis("off")
plt.show()

### 4. RANSAC for outlier rejection and homography estimation

## RANSAC visualization

The following figure illustrates RANSAC results.

In [ ]:
# Ransac to filter out the wrong matchings and return homography
def RANSAC(match_kp1, match_kp2, N, t, threshold):
    max_inliers = []
    best_points1 = []
    best_points2 = []
    best_H = None
    k = 0

    while (k < N) and (len(max_inliers) < threshold):
        # print(f"i = {k} //// {len(max_inliers)} out of {threshold} inliers", flush=True)
        points1 = []
        points2 = []
        inliers = []

        # Choose four feature pairs at random
        ind = np.random.choice(len(match_kp1), size=4, replace=False)

        for i in ind:
            points1.append(match_kp1[i].pt)
            points2.append(match_kp2[i].pt)

        # Compute the homography matrix h using the selected points
        h = cv2.getPerspectiveTransform(np.float32(points1), np.float32(points2))

        # Compute inliers
        if h is not None:

            for j in range(len(match_kp1)):
                # Convert into homogeneous coordinates
                point1 = np.append(np.float32(match_kp1[j].pt), 1).reshape(3, 1)
                # Compute Hpi
                projected_point = np.dot(h, point1)
                # Avoiding divide by zero
                if projected_point[2] > 0:
                    # Convert into Cartesian coordinates
                    projected_point = projected_point / projected_point[2]
                    projected_point = projected_point[:2].reshape(-1)
                    # Compute ssd
                    ssd = np.sum((match_kp2[j].pt - projected_point) ** 2)
                    # Add inliers
                    if ssd < t:
                        inliers.append((j, ssd))

        # Check if current inlier count is the largest so far
        if len(inliers) > len(max_inliers):
            max_inliers = inliers

        k = k + 1

    # Compute homography on 4 edges with least SSD (According to TA)
    sorted_inliers = sorted(max_inliers, key=lambda x: x[1])

    for i, sdd in sorted_inliers[:4]:
        best_points1.append(match_kp1[i].pt)
        best_points2.append(match_kp2[i].pt)

    best_H = cv2.getPerspectiveTransform(
        np.float32(best_points1), np.float32(best_points2)
    )

    return best_H, max_inliers

In [ ]:
# Estimate a robust homography with RANSAC and visualize inliers

# get feature descriptors for corners in image 1
img1 = cv2.imread(f"{DATA_DIR}/TestSet1/2.jpg")
initial_corners1, cmap1 = detect_corner(img1)
corners1 = ANMS(cmap1, initial_corners1, 100)
features1 = feature_descript(cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY), corners1)

# get feature descriptors for corners in image 2
img2 = cv2.imread(f"{DATA_DIR}/TestSet1/3.jpg")
initial_corners2, cmap2 = detect_corner(img2)
corners2 = ANMS(cmap2, initial_corners2, 100)
features2 = feature_descript(cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY), corners2)

# run feature_match
matches = feature_match(-1, -1, features1, features2)

# convert corners into key point objects
kp1 = []
for x, y in corners1:
    kp1.append(cv2.KeyPoint(y.astype(float), x.astype(float), 1))

# convert corners into key point objects
kp2 = []
for x, y in corners2:
    kp2.append(cv2.KeyPoint(y.astype(float), x.astype(float), 1))

# Extract the matched points
match_kp1 = []
match_kp2 = []
for m in matches:
    match_kp1.append(kp1[m.queryIdx])
    match_kp2.append(kp2[m.trainIdx])

# RANSAC
N = 1000
t = 7
threshold = int(np.ceil(len(matches) * 0.9))  # 90% inliers threshold
best_h, inliers = RANSAC(match_kp1, match_kp2, N, t, threshold)

# Draw only inlier matches
inlier_matches = [matches[i] for i, ssd in inliers]
print(f"{len(inlier_matches)} inliers")
# Visualize the inliers
out = cv2.drawMatches(
    cv2.cvtColor(img1, cv2.COLOR_BGR2RGB),
    kp1,
    cv2.cvtColor(img2, cv2.COLOR_BGR2RGB),
    kp2,
    inlier_matches,
    None,
)


plt.figure(figsize=(16, 12))
plt.imshow(out)
plt.axis("off")
plt.show()

### 5. Image Warping and Blending

Panoramas are produced by warping a source image onto a reference canvas via the estimated homography and blending the overlap into a single mosaic.

Photometric differences between images (exposure, white balance, vignetting) can cause visible seams. To reduce these, we blend in the overlap region. OpenCV's seamlessClone (Poisson blending) is used here as a simple, off-the-shelf approach.

In [ ]:
# Warp and Blend two images together based on Homography returned in RANSAC
def warp_and_blend(img1, img2, H):
    inv_h = np.linalg.inv(H)  # Inverse to get hg from img2 to img1

    # Get the shape of the images
    h1, w1 = img1.shape[:2]  # Size of img1
    h2, w2 = img2.shape[:2]  # Size of img2

    # Define corners of img2
    corners_img2 = np.array(
        [[0, 0], [w2, 0], [w2, h2], [0, h2]], dtype=np.float32
    ).reshape(-1, 1, 2)

    # Warp the corners of img2 using the homography
    warped_corners_img2 = cv2.perspectiveTransform(corners_img2, inv_h)

    # Define the corners of img1 (which is at (0,0) in the panorama space)
    corners_img1 = np.array(
        [[0, 0], [w1, 0], [w1, h1], [0, h1]], dtype=np.float32
    ).reshape(-1, 1, 2)

    # Combine all corners (from warped img2 and img1) to calculate the bounding box
    all_corners = np.vstack((warped_corners_img2, corners_img1))

    # Find the bounding box for the panorama
    x_min, y_min = np.int32(all_corners.min(axis=0).ravel())
    x_max, y_max = np.int32(all_corners.max(axis=0).ravel())

    # Calculate the size of the output panorama
    output_width = x_max - x_min
    output_height = y_max - y_min

    # Create a translation matrix to move everything to the positive quadrant (if x_min or y_min < 0)
    translation_matrix = np.array([[1, 0, -x_min], [0, 1, -y_min], [0, 0, 1]])

    # Warp img2 to the new size and translation
    warped_img = cv2.warpPerspective(
        img2, translation_matrix @ inv_h, (output_width, output_height)
    )

    # Create a mask for img2
    mask = np.zeros(warped_img.shape, dtype=np.uint8)
    mask[warped_img > 0] = 255
    mask = mask[:, :, 0]

    # Enlarge img1 by padding with zeros (black) to match the size of the panorama
    img1_padded = np.zeros((output_height, output_width, 3), dtype=img1.dtype)
    img1_padded[-y_min : h1 - y_min, -x_min : w1 - x_min] = img1

    # Create a mask for img1 that identifies non-black pixels (valid region in warped_img2)
    mask_img2 = cv2.cvtColor(warped_img, cv2.COLOR_BGR2GRAY)
    _, mask_img2 = cv2.threshold(mask_img2, 1, 255, cv2.THRESH_BINARY)

    # Create a mask for img2 that identifies non-black pixels (valid region in img2_padded)
    mask_img1 = cv2.cvtColor(img1_padded, cv2.COLOR_BGR2GRAY)
    _, mask_img1 = cv2.threshold(mask_img1, 1, 255, cv2.THRESH_BINARY)

    # Combine the two masks to find the overlapping region (where both masks are 255)
    overlap_mask = cv2.bitwise_and(mask_img1, mask_img2)
    plt.imshow(overlap_mask)
    plt.show()

    # Find the non-overlapping region of img2 using a mask
    non_overlap_mask = cv2.bitwise_xor(mask_img2, overlap_mask)
    plt.imshow(non_overlap_mask)
    plt.show()

    # Add the non-overlapping region of img2 to img1_padded
    non_overlap_region = cv2.bitwise_and(warped_img, warped_img, mask=non_overlap_mask)
    img1_padded = cv2.add(img1_padded, non_overlap_region)
    plt.imshow(img1_padded)
    plt.show()

    # Find the center point for seamlessClone (center of the warped image)
    center_x = int(
        (warped_corners_img2[:, :, 0].min() + warped_corners_img2[:, :, 0].max()) / 2
        - x_min
    )
    center_y = int(
        (warped_corners_img2[:, :, 1].min() + warped_corners_img2[:, :, 1].max()) / 2
        - y_min
    )
    center = (center_x, center_y)
    print(center)

    # Perform seamless cloning using NORMAL_CLONE
    blend_img = cv2.seamlessClone(
        warped_img, img1_padded, mask, center, cv2.NORMAL_CLONE
    )
    # blend_img = img1_padded

    return blend_img

In [ ]:
# NEW WARP + BLEND (returns blended image and translation)

def warp_and_blend_with_transform(img1, img2, inv_h):

    # Get the shape of the images
    h1, w1 = img1.shape[:2]  # Size of img1
    h2, w2 = img2.shape[:2]  # Size of img2

    # Define corners of img2
    corners_img2 = np.array(
        [[0, 0], [w2, 0], [w2, h2], [0, h2]], dtype=np.float32
    ).reshape(-1, 1, 2)

    # Warp the corners of img2 using the homography
    warped_corners_img2 = cv2.perspectiveTransform(corners_img2, inv_h)

    # Define the corners of img1 (which is at (0,0) in the panorama space)
    corners_img1 = np.array(
        [[0, 0], [w1, 0], [w1, h1], [0, h1]], dtype=np.float32
    ).reshape(-1, 1, 2)

    # Combine all corners (from warped img2 and img1) to calculate the bounding box
    all_corners = np.vstack((warped_corners_img2, corners_img1))

    # Find the bounding box for the panorama
    x_min, y_min = np.int32(all_corners.min(axis=0).ravel())
    x_max, y_max = np.int32(all_corners.max(axis=0).ravel())

    # Calculate the size of the output panorama
    output_width = x_max - x_min
    output_height = y_max - y_min

    # Create a translation matrix to move everything to the positive quadrant (if x_min or y_min < 0)
    translation_matrix = np.array([[1, 0, -x_min], [0, 1, -y_min], [0, 0, 1]])

    # Warp img2 to the new size and translation
    warped_img = cv2.warpPerspective(
        img2, translation_matrix @ inv_h, (output_width, output_height)
    )

    # Create a mask for img2
    mask = np.zeros(warped_img.shape, dtype=np.uint8)
    mask[warped_img > 0] = 255
    mask = mask[:, :, 0]

    # Enlarge img1 by padding with zeros (black) to match the size of the panorama
    img1_padded = np.zeros((output_height, output_width, 3), dtype=img1.dtype)
    img1_padded[-y_min : h1 - y_min, -x_min : w1 - x_min] = img1

    # Create a mask for img1 that identifies non-black pixels (valid region in warped_img2)
    mask_img2 = cv2.cvtColor(warped_img, cv2.COLOR_BGR2GRAY)
    _, mask_img2 = cv2.threshold(mask_img2, 1, 255, cv2.THRESH_BINARY)

    # Create a mask for img2 that identifies non-black pixels (valid region in img2_padded)
    mask_img1 = cv2.cvtColor(img1_padded, cv2.COLOR_BGR2GRAY)
    _, mask_img1 = cv2.threshold(mask_img1, 1, 255, cv2.THRESH_BINARY)

    # Combine the two masks to find the overlapping region (where both masks are 255)
    overlap_mask = cv2.bitwise_and(mask_img1, mask_img2)
    plt.imshow(overlap_mask)
    plt.show()

    # Find the non-overlapping region of img2 using a mask
    non_overlap_mask = cv2.bitwise_xor(mask_img2, overlap_mask)
    plt.imshow(non_overlap_mask)
    plt.show()

    # Add the non-overlapping region of img2 to img1_padded
    non_overlap_region = cv2.bitwise_and(warped_img, warped_img, mask=non_overlap_mask)
    img1_padded = cv2.add(img1_padded, non_overlap_region)
    plt.imshow(img1_padded)
    plt.show()

    # Find the center point for seamlessClone (center of the warped image)
    center_x = int(
        (warped_corners_img2[:, :, 0].min() + warped_corners_img2[:, :, 0].max()) / 2
        - x_min
    )
    center_y = int(
        (warped_corners_img2[:, :, 1].min() + warped_corners_img2[:, :, 1].max()) / 2
        - y_min
    )
    center = (center_x, center_y)
    print(center)

    # Perform seamless cloning using NORMAL_CLONE
    blend_img = cv2.seamlessClone(
        warped_img, img1_padded, mask, center, cv2.NORMAL_CLONE
    )
    # blend_img = img1_padded

    return blend_img, translation_matrix

In [ ]:
# get feature descriptors for corners in image 1
img1 = cv2.imread(f"{DATA_DIR}/TestSet1/1.jpg")
initial_corners1, cmap1 = detect_corner(img1)
corners1 = ANMS(cmap1, initial_corners1, 100)
features1 = feature_descript(cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY), corners1)

# get feature descriptors for corners in image 2
img2 = cv2.imread(f"{DATA_DIR}/TestSet1/2.jpg")
initial_corners2, cmap2 = detect_corner(img2)
corners2 = ANMS(cmap2, initial_corners2, 100)
features2 = feature_descript(cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY), corners2)

# run feature_match
matches = feature_match(-1, -1, features1, features2)

# convert corners into key point objects
kp1 = []
for x, y in corners1:
    kp1.append(cv2.KeyPoint(y.astype(float), x.astype(float), 1))

# convert corners into key point objects
kp2 = []
for x, y in corners2:
    kp2.append(cv2.KeyPoint(y.astype(float), x.astype(float), 1))

# Extract the matched points
match_kp1 = []
match_kp2 = []
for m in matches:
    match_kp1.append(kp1[m.queryIdx])
    match_kp2.append(kp2[m.trainIdx])

# RANSAC
N = 1000
t = 7
threshold = int(np.ceil(len(matches) * 0.9))  # 90% inliers threshold
best_h, inliers = RANSAC(match_kp1, match_kp2, N, t, threshold)
print(best_h)
# Show you result here
result = warp_and_blend(img1, img2, best_h)
# sharpen_kernel = np.array([[-1, -1, -1],
#                            [-1,  9, -1],
#                            [-1, -1, -1]])
# sharpened_image = cv2.filter2D(result, -1, sharpen_kernel)
# sharp_img = cv2.cvtColor(sharpened_image, cv2.COLOR_BGR2RGB)
# plt.imshow(sharp_img)
# plt.axis("off")
# plt.show()
result = cv2.cvtColor(result, cv2.COLOR_BGR2RGB)
plt.imshow(result)
plt.axis("off")
plt.show()

In [ ]:
plt.imshow(result)
plt.axis("off")
plt.show()
result2 = cv2.cvtColor(result, cv2.COLOR_RGB2BGR)
cv2.imwrite(f"{RESULTS_DIR}/result2.jpg", result2)

In [ ]:
img2 = cv2.imread(f"{RESULTS_DIR}/result2.jpg")
img2 = cv2.cvtColor(img2, cv2.COLOR_BGR2RGB)
plt.imshow(img2)
plt.axis("off")
plt.show()

In [ ]:
# MULTIPLYING HOMOGRAPHIES TEST

# COMPUTE HOMOGRAPHY OF IMG1 TO IMG2
# get feature descriptors for corners in image 1
img3 = cv2.imread(f"{DATA_DIR}/TestSet1/1.jpg")
initial_corners3, cmap3 = detect_corner(img3)
corners3 = ANMS(cmap3, initial_corners3, 100)
features3 = feature_descript(cv2.cvtColor(img3, cv2.COLOR_BGR2GRAY), corners3)

# get feature descriptors for corners in image 2
img4 = cv2.imread(f"{DATA_DIR}/TestSet1/2.jpg")
initial_corners4, cmap4 = detect_corner(img4)
corners4 = ANMS(cmap4, initial_corners4, 100)
features4 = feature_descript(cv2.cvtColor(img4, cv2.COLOR_BGR2GRAY), corners4)

# run feature_match
matches = feature_match(-1, -1, features3, features4)

# convert corners into key point objects
kp1 = []
for x, y in corners3:
    kp1.append(cv2.KeyPoint(y.astype(float), x.astype(float), 1))

# convert corners into key point objects
kp2 = []
for x, y in corners4:
    kp2.append(cv2.KeyPoint(y.astype(float), x.astype(float), 1))

# Extract the matched points
match_kp1 = []
match_kp2 = []
for m in matches:
    match_kp1.append(kp1[m.queryIdx])
    match_kp2.append(kp2[m.trainIdx])

# RANSAC
N = 1000
t = 7
threshold = int(np.ceil(len(matches) * 0.9))  # 90% inliers threshold
print(len(match_kp1))
h1to2, inliers = RANSAC(match_kp1, match_kp2, N, t, threshold)

# COMPUTE HOMOGRAPHY OF IMG2 TO IMG3
# get feature descriptors for corners in image 1
img3 = cv2.imread(f"{DATA_DIR}/TestSet1/2.jpg")
initial_corners3, cmap3 = detect_corner(img3)
corners3 = ANMS(cmap3, initial_corners3, 100)
features3 = feature_descript(cv2.cvtColor(img3, cv2.COLOR_BGR2GRAY), corners3)

# get feature descriptors for corners in image 2
img4 = cv2.imread(f"{DATA_DIR}/TestSet1/3.jpg")
initial_corners4, cmap4 = detect_corner(img4)
corners4 = ANMS(cmap4, initial_corners4, 100)
features4 = feature_descript(cv2.cvtColor(img4, cv2.COLOR_BGR2GRAY), corners4)

# run feature_match
matches = feature_match(-1, -1, features3, features4)

# convert corners into key point objects
kp1 = []
for x, y in corners3:
    kp1.append(cv2.KeyPoint(y.astype(float), x.astype(float), 1))

# convert corners into key point objects
kp2 = []
for x, y in corners4:
    kp2.append(cv2.KeyPoint(y.astype(float), x.astype(float), 1))

# Extract the matched points
match_kp1 = []
match_kp2 = []
for m in matches:
    match_kp1.append(kp1[m.queryIdx])
    match_kp2.append(kp2[m.trainIdx])

# RANSAC
N = 1000
t = 7
threshold = int(np.ceil(len(matches) * 0.9))  # 90% inliers threshold
print(len(match_kp1))
h2to3, inliers = RANSAC(match_kp1, match_kp2, N, t, threshold)


# WARP AND BLEND IMG2 INTO IMG1
img1 = cv2.imread(f"{DATA_DIR}/TestSet1/1.jpg")
img2 = cv2.imread(f"{DATA_DIR}/TestSet1/2.jpg")

# inv_best_h = np.linalg.inv(best_h) # 2 to 1
# inv_h2to3 = np.linalg.inv(h2to3) # 3 to 2
# inv_h = np.linalg.inv(best_h @ h2to3) # 3 to 1
inv_h = np.linalg.inv(h1to2)  # Inverse to get hg from img2 to img1
print(f"FIRST inv_h: {inv_h}")

# Get the shape of the images
h1, w1 = img1.shape[:2]  # Size of img1
h2, w2 = img2.shape[:2]  # Size of img2

# Define corners of img2
corners_img2 = np.array([[0, 0], [w2, 0], [w2, h2], [0, h2]], dtype=np.float32).reshape(
    -1, 1, 2
)

# Warp the corners of img2 using the homography
warped_corners_img2 = cv2.perspectiveTransform(corners_img2, inv_h)

# Define the corners of img1 (which is at (0,0) in the panorama space)
corners_img1 = np.array([[0, 0], [w1, 0], [w1, h1], [0, h1]], dtype=np.float32).reshape(
    -1, 1, 2
)

# Combine all corners (from warped img2 and img1) to calculate the bounding box
all_corners = np.vstack((warped_corners_img2, corners_img1))

# Find the bounding box for the panorama
x_min, y_min = np.int32(all_corners.min(axis=0).ravel())
x_max, y_max = np.int32(all_corners.max(axis=0).ravel())

# Calculate the size of the output panorama
output_width = x_max - x_min
output_height = y_max - y_min

# Create a translation matrix to move everything to the positive quadrant (if x_min or y_min < 0)
translation_matrix1 = np.array([[1, 0, -x_min], [0, 1, -y_min], [0, 0, 1]])
print(f"FIRST T: {translation_matrix1}")

# Warp img2 to the new size and translation
warped_img = cv2.warpPerspective(
    img2, translation_matrix1 @ inv_h, (output_width, output_height)
)

# Create a mask for img2
mask = np.zeros(warped_img.shape, dtype=np.uint8)
mask[warped_img > 0] = 255
mask = mask[:, :, 0]

# Enlarge img1 by padding with zeros (black) to match the size of the panorama
img1_padded = np.zeros((output_height, output_width, 3), dtype=img1.dtype)
img1_padded[-y_min : h1 - y_min, -x_min : w1 - x_min] = img1

# Create a mask for img1 that identifies non-black pixels (valid region in warped_img2)
mask_img2 = cv2.cvtColor(warped_img, cv2.COLOR_BGR2GRAY)
_, mask_img2 = cv2.threshold(mask_img2, 1, 255, cv2.THRESH_BINARY)

# Create a mask for img2 that identifies non-black pixels (valid region in img2_padded)
mask_img1 = cv2.cvtColor(img1_padded, cv2.COLOR_BGR2GRAY)
_, mask_img1 = cv2.threshold(mask_img1, 1, 255, cv2.THRESH_BINARY)

# Combine the two masks to find the overlapping region (where both masks are 255)
overlap_mask = cv2.bitwise_and(mask_img1, mask_img2)
plt.imshow(overlap_mask)
plt.show()

# Find the non-overlapping region of img2 using a mask
non_overlap_mask = cv2.bitwise_xor(mask_img2, overlap_mask)
plt.imshow(non_overlap_mask)
plt.show()

# Add the non-overlapping region of img2 to img1_padded
non_overlap_region = cv2.bitwise_and(warped_img, warped_img, mask=non_overlap_mask)
img1_padded = cv2.add(img1_padded, non_overlap_region)
plt.imshow(img1_padded)
plt.show()

# Find the center point for seamlessClone (center of the warped image)
center_x = int(
    (warped_corners_img2[:, :, 0].min() + warped_corners_img2[:, :, 0].max()) / 2
    - x_min
)
center_y = int(
    (warped_corners_img2[:, :, 1].min() + warped_corners_img2[:, :, 1].max()) / 2
    - y_min
)
center = (center_x, center_y)
print(center)

# Perform seamless cloning using NORMAL_CLONE
blend_img = cv2.seamlessClone(warped_img, img1_padded, mask, center, cv2.NORMAL_CLONE)

plt.imshow(blend_img)
plt.axis("off")
plt.show()


# WARP AND BLEND IMG3 INTO IMG1AND2
img1 = blend_img
img2 = img4

inv_best_h = np.linalg.inv(h1to2)  # 2 to 1
inv_h2to3 = np.linalg.inv(h2to3)  # 3 to 2
inv_h = inv_best_h @ inv_h2to3  # 3 to 1
inv_h = translation_matrix1 @ inv_h
print(f"SECOND inv_h: {inv_h}")
# inv_h = np.linalg.inv(H) # Inverse to get hg from img2 to img1

# Get the shape of the images
h1, w1 = img1.shape[:2]  # Size of img1
h2, w2 = img2.shape[:2]  # Size of img2

# Define corners of img2
corners_img2 = np.array([[0, 0], [w2, 0], [w2, h2], [0, h2]], dtype=np.float32).reshape(
    -1, 1, 2
)

# Warp the corners of img2 using the homography
warped_corners_img2 = cv2.perspectiveTransform(corners_img2, inv_h)

# Define the corners of img1 (which is at (0,0) in the panorama space)
corners_img1 = np.array([[0, 0], [w1, 0], [w1, h1], [0, h1]], dtype=np.float32).reshape(
    -1, 1, 2
)

# Combine all corners (from warped img2 and img1) to calculate the bounding box
all_corners = np.vstack((warped_corners_img2, corners_img1))

# Find the bounding box for the panorama
x_min, y_min = np.int32(all_corners.min(axis=0).ravel())
x_max, y_max = np.int32(all_corners.max(axis=0).ravel())

# Calculate the size of the output panorama
output_width = x_max - x_min
output_height = y_max - y_min

# Create a translation matrix to move everything to the positive quadrant (if x_min or y_min < 0)
translation_matrix2 = np.array([[1, 0, -x_min], [0, 1, -y_min], [0, 0, 1]])

# Warp img2 to the new size and translation
warped_img = cv2.warpPerspective(
    img2, translation_matrix2 @ inv_h, (output_width, output_height)
)

# Create a mask for img2
mask = np.zeros(warped_img.shape, dtype=np.uint8)
mask[warped_img > 0] = 255
mask = mask[:, :, 0]

# Enlarge img1 by padding with zeros (black) to match the size of the panorama
img1_padded = np.zeros((output_height, output_width, 3), dtype=img1.dtype)
img1_padded[-y_min : h1 - y_min, -x_min : w1 - x_min] = img1

# Create a mask for img1 that identifies non-black pixels (valid region in warped_img2)
mask_img2 = cv2.cvtColor(warped_img, cv2.COLOR_BGR2GRAY)
_, mask_img2 = cv2.threshold(mask_img2, 1, 255, cv2.THRESH_BINARY)

# Create a mask for img2 that identifies non-black pixels (valid region in img2_padded)
mask_img1 = cv2.cvtColor(img1_padded, cv2.COLOR_BGR2GRAY)
_, mask_img1 = cv2.threshold(mask_img1, 1, 255, cv2.THRESH_BINARY)

# Combine the two masks to find the overlapping region (where both masks are 255)
overlap_mask = cv2.bitwise_and(mask_img1, mask_img2)
plt.imshow(overlap_mask)
plt.show()

# Find the non-overlapping region of img2 using a mask
non_overlap_mask = cv2.bitwise_xor(mask_img2, overlap_mask)
plt.imshow(non_overlap_mask)
plt.show()

# Add the non-overlapping region of img2 to img1_padded
non_overlap_region = cv2.bitwise_and(warped_img, warped_img, mask=non_overlap_mask)
img1_padded = cv2.add(img1_padded, non_overlap_region)
plt.imshow(img1_padded)
plt.show()

# Find the center point for seamlessClone (center of the warped image)
center_x = int(
    (warped_corners_img2[:, :, 0].min() + warped_corners_img2[:, :, 0].max()) / 2
    - x_min
)
center_y = int(
    (warped_corners_img2[:, :, 1].min() + warped_corners_img2[:, :, 1].max()) / 2
    - y_min
)
center = (center_x, center_y)
print(center)

# Perform seamless cloning using NORMAL_CLONE
blend_img = cv2.seamlessClone(warped_img, img1_padded, mask, center, cv2.NORMAL_CLONE)

plt.imshow(blend_img)
plt.axis("off")
plt.show()

In [ ]:
# get feature descriptors for corners in image 1
img3 = cv2.imread(f"{RESULTS_DIR}/result2.jpg")
initial_corners3, cmap3 = detect_corner(img3)
corners3 = ANMS(cmap3, initial_corners3, 100)
features3 = feature_descript(cv2.cvtColor(img3, cv2.COLOR_BGR2GRAY), corners3)

# get feature descriptors for corners in image 2
img4 = cv2.imread(f"{DATA_DIR}/TestSet1/1.jpg")
initial_corners4, cmap4 = detect_corner(img4)
corners4 = ANMS(cmap4, initial_corners4, 100)
features4 = feature_descript(cv2.cvtColor(img4, cv2.COLOR_BGR2GRAY), corners4)

# run feature_match
matches = feature_match(-1, -1, features3, features4)

# convert corners into key point objects
kp1 = []
for x, y in corners3:
    kp1.append(cv2.KeyPoint(y.astype(float), x.astype(float), 1))

# convert corners into key point objects
kp2 = []
for x, y in corners4:
    kp2.append(cv2.KeyPoint(y.astype(float), x.astype(float), 1))

# Extract the matched points
match_kp1 = []
match_kp2 = []
for m in matches:
    match_kp1.append(kp1[m.queryIdx])
    match_kp2.append(kp2[m.trainIdx])

# RANSAC
N = 1000
t = 7
threshold = int(np.ceil(len(matches) * 0.9))  # 90% inliers threshold
print(len(match_kp1))
best_h, inliers = RANSAC(match_kp1, match_kp2, N, t, threshold)

# Show you result here
result = warp_and_blend(img3, img4, best_h)
plt.imshow(result)
plt.axis("off")
plt.show()

In [ ]:
# get feature descriptors for corners in image 1
img1 = cv2.imread(f"{RESULTS_DIR}/result2.jpg")
initial_corners1, cmap1 = detect_corner(img1)
corners1 = ANMS(cmap1, initial_corners1, 100)
features1 = feature_descript(cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY), corners1)

# get feature descriptors for corners in image 2
img2 = cv2.imread(f"{DATA_DIR}/TestSet2/3.jpg")
initial_corners2, cmap2 = detect_corner(img2)
corners2 = ANMS(cmap2, initial_corners2, 100)
features2 = feature_descript(cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY), corners2)

# run feature_match
matches = feature_match(-1, -1, features1, features2)

# convert corners into key point objects
kp1 = []
for x, y in corners1:
    kp1.append(cv2.KeyPoint(y.astype(float), x.astype(float), 1))

# convert corners into key point objects
kp2 = []
for x, y in corners2:
    kp2.append(cv2.KeyPoint(y.astype(float), x.astype(float), 1))

# Extract the matched points
match_kp1 = []
match_kp2 = []
for m in matches:
    match_kp1.append(kp1[m.queryIdx])
    match_kp2.append(kp2[m.trainIdx])

# RANSAC
N = 1000
t = 7
threshold = int(np.ceil(len(matches) * 0.9))  # 90% inliers threshold
best_h, inliers = RANSAC(match_kp1, match_kp2, N, t, threshold)

# Show you result here

# Draw only inlier matches
inlier_matches = [matches[i] for i, ssd in inliers]

# Visualize the inliers
out = cv2.drawMatches(
    cv2.cvtColor(img1, cv2.COLOR_BGR2RGB),
    kp1,
    cv2.cvtColor(img2, cv2.COLOR_BGR2RGB),
    kp2,
    inlier_matches,
    None,
)


plt.figure(figsize=(16, 12))
plt.imshow(out)
plt.axis("off")
plt.show()

In [ ]:
N = 1000
t = 7
threshold = int(np.ceil(len(matches) * 0.9))  # 90% inliers threshold
best_h, inliers = RANSAC(match_kp1, match_kp2, N, t, threshold)

# Show you result here
inv_h = np.linalg.inv(best_h)  # Inverse to get hg from img2 to img1

# Get the shape of the images
h1, w1 = img1.shape[:2]  # Size of img1
h2, w2 = img2.shape[:2]  # Size of img2
print(h1, w1)
print(h2, w2)

# Define corners of img2
corners_img2 = np.array([[0, 0], [w2, 0], [w2, h2], [0, h2]], dtype=np.float32).reshape(
    -1, 1, 2
)
print(corners_img2)

# Warp the corners of img2 using the homography
warped_corners_img2 = cv2.perspectiveTransform(corners_img2, inv_h)

# Define the corners of img1 (which is at (0,0) in the panorama space)
corners_img1 = np.array([[0, 0], [w1, 0], [w1, h1], [0, h1]], dtype=np.float32).reshape(
    -1, 1, 2
)

# Combine all corners (from warped img2 and img1) to calculate the bounding box
all_corners = np.vstack((warped_corners_img2, corners_img1))

# Find the bounding box for the panorama
x_min, y_min = np.int32(all_corners.min(axis=0).ravel())
x_max, y_max = np.int32(all_corners.max(axis=0).ravel())
print(x_min)
print(x_max)
print(y_min)
print(y_max)

# Calculate the size of the output panorama
output_width = x_max - x_min
output_height = y_max - y_min

# Create a translation matrix to move everything to the positive quadrant (if x_min or y_min < 0)
translation_matrix = np.array([[1, 0, -x_min], [0, 1, -y_min], [0, 0, 1]])

# Warp img2 to the new size and translation
warped_img = cv2.warpPerspective(
    img2, translation_matrix @ inv_h, (output_width, output_height)
)

# Create a mask for img (all white)
mask = np.zeros(warped_img.shape, dtype=np.uint8)
mask[warped_img > 0] = 255
mask = mask[:, :, 0]

# Enlarge img1 by padding with zeros (black) to match the size of the panorama
img1_padded = np.zeros((output_height, output_width, 3), dtype=img1.dtype)
img1_padded[-y_min : h1 - y_min, -x_min : w1 - x_min] = img1


# Find the center point for seamlessClone (center of the warped image)
center_x = int(
    (warped_corners_img2[:, :, 0].min() + warped_corners_img2[:, :, 0].max()) / 2
    - x_min
)
center_y = int(
    (warped_corners_img2[:, :, 1].min() + warped_corners_img2[:, :, 1].max()) / 2
    - y_min
)
center = (center_x, center_y)
print(center)

plt.imshow(img1_padded)
plt.show()
plt.imshow(warped_img)
plt.show()
plt.imshow(mask)
plt.show()
# Perform seamless cloning using NORMAL_CLONE
blend_img = cv2.seamlessClone(warped_img, img1_padded, mask, center, cv2.NORMAL_CLONE)

# plt.imshow(img1)
# plt.show()
# plt.imshow(img2)
# plt.show()
# plt.imshow(warped_img)
# plt.show()
plt.imshow(blend_img)
plt.show()

### 6. Putting Everything Together (Stitching more than 2 images)

In [ ]:
# Take in a list of images and stitch them together!
def pano_imgs(img_list):
    main_img = img_list[0]

    for next_img in img_list[1:]:
        # get feature descriptors for corners in main_image
        initial_corners1, cmap1 = detect_corner(main_img)
        corners1 = ANMS(cmap1, initial_corners1, 100)
        features1 = feature_descript(
            cv2.cvtColor(main_img, cv2.COLOR_BGR2GRAY), corners1
        )

        # get feature descriptors for corners in next img
        initial_corners2, cmap2 = detect_corner(next_img)
        corners2 = ANMS(cmap2, initial_corners2, 100)
        features2 = feature_descript(
            cv2.cvtColor(next_img, cv2.COLOR_BGR2GRAY), corners2
        )

        # run feature_match
        matches = feature_match(-1, -1, features1, features2)

        # convert corners into key point objects
        kp1 = []
        for x, y in corners1:
            kp1.append(cv2.KeyPoint(y.astype(float), x.astype(float), 1))

        # convert corners into key point objects
        kp2 = []
        for x, y in corners2:
            kp2.append(cv2.KeyPoint(y.astype(float), x.astype(float), 1))

        # Extract the matched points
        match_kp1 = []
        match_kp2 = []
        for m in matches:
            match_kp1.append(kp1[m.queryIdx])
            match_kp2.append(kp2[m.trainIdx])

        # RANSAC
        N = 1000
        t = 7
        threshold = int(np.ceil(len(matches) * 0.9))  # 90% inliers threshold
        best_h, inliers = RANSAC(match_kp1, match_kp2, N, t, threshold)

        # Warp and Blend into one image
        main_img = warp_and_blend(main_img, next_img, best_h)

        # # Turn back into BGR for input
        # main_img = cv2.cvtColor(main_img, cv2.COLOR_RGB2BGR)

    return main_img

In [ ]:
def pano_imgs(img_list):
    # Compute inverse homography for each image
    inv_H_list = []
    current_img = img_list[0]

    for next_img in img_list[1:]:
        # get feature descriptors for corners in current_img
        initial_corners1, cmap1 = detect_corner(current_img)
        corners1 = ANMS(cmap1, initial_corners1, 100)
        features1 = feature_descript(
            cv2.cvtColor(current_img, cv2.COLOR_BGR2GRAY), corners1
        )

        # get feature descriptors for corners in next img
        initial_corners2, cmap2 = detect_corner(next_img)
        corners2 = ANMS(cmap2, initial_corners2, 100)
        features2 = feature_descript(
            cv2.cvtColor(next_img, cv2.COLOR_BGR2GRAY), corners2
        )

        # run feature_match
        matches = feature_match(-1, -1, features1, features2)

        # convert corners into key point objects
        kp1 = []
        for x, y in corners1:
            kp1.append(cv2.KeyPoint(y.astype(float), x.astype(float), 1))

        # convert corners into key point objects
        kp2 = []
        for x, y in corners2:
            kp2.append(cv2.KeyPoint(y.astype(float), x.astype(float), 1))

        # Extract the matched points
        match_kp1 = []
        match_kp2 = []
        for m in matches:
            match_kp1.append(kp1[m.queryIdx])
            match_kp2.append(kp2[m.trainIdx])

        # RANSAC
        N = 1000
        t = 7
        threshold = int(np.ceil(len(matches) * 0.9))  # 90% inliers threshold
        best_h, inliers = RANSAC(match_kp1, match_kp2, N, t, threshold)

        inv_H_list.append(np.linalg.inv(best_h))
        current_img = next_img

    # Use inv_H_list to warp and blend
    main_img = img_list[0]
    translations = []

    for i in range(1, len(img_list)):
        current_h = np.eye(3)
        current_t = np.eye(3)

        # Multiply all previous homographies
        for j in range(i):
            # print(f"inv_H {j} BEFORE @ {inv_H_list[j]}")
            current_h = current_h @ inv_H_list[j]

        # Multiple all previous translations
        for k in translations:
            print(f"FIRST T: {k}")
            current_t = k @ current_t

        # Multiply translation into homography
        current_h = current_t @ current_h
        print(f"{i} inv_h: {current_h}")
        # print(f"inv_H AFTER @ {current_h}")
        # Warp and blend
        main_img, new_t = warp_and_blend_with_transform(main_img, img_list[i], current_h)
        translations.append(new_t)
        plt.imshow(main_img)
        plt.show()

    return main_img

In [ ]:
import glob
import re


# Sort images to read them in order
def extract_number(path):
    # Extract the filename part and use regex to find the number in the image name
    match = re.search(r"(\d+)\.jpg", path)
    return int(match.group(1)) if match else 0


def middle_left_right_order(images):
    result = []
    n = len(images)
    middle = n // 2  # Find the index of the middle image

    # Add the middle image first
    result.append(images[middle])

    # Alternate between left and right of the middle
    left = middle - 1
    right = middle + 1

    while left >= 0 or right < n:
        if right < n:
            result.append(images[right])
            right += 1
        if left >= 0:
            result.append(images[left])
            left -= 1

    return result


# Run stitching on TestSet1
img_list = []
folder_path = f"{DATA_DIR}/TestSet1/"
image_paths = glob.glob(folder_path + "*.jpg")
image_paths = sorted(image_paths, key=extract_number)
# image_paths = middle_left_right_order(image_paths)

for path in image_paths:
    img_list.append(cv2.imread(path))
print(image_paths)

result = pano_imgs(img_list)
result = cv2.cvtColor(result, cv2.COLOR_BGR2RGB)
plt.imshow(result)
plt.axis("off")
plt.show()

In [ ]:
img_list = []
folder_path = f"{DATA_DIR}/TestSet2/"
image_paths = glob.glob(folder_path + "*.jpg")
image_paths = sorted(image_paths, key=extract_number)
# image_paths = middle_left_right_order(image_paths)

for path in image_paths:
    img_list.append(cv2.imread(path))
print(image_paths)

result = pano_imgs(img_list)
result = cv2.cvtColor(result, cv2.COLOR_BGR2RGB)
plt.imshow(result)
plt.axis("off")
plt.show()

In [ ]:
img_list = []
folder_path = f"{DATA_DIR}/TestSet3/"
image_paths = glob.glob(folder_path + "*.jpg")
image_paths = sorted(image_paths, key=extract_number)
# image_paths = middle_left_right_order(image_paths)

for path in image_paths:
    img_list.append(cv2.imread(path))
print(image_paths)

# max_count = 100
# count = 0
# while count < max_count:
#   try:
#     result = pano_imgs(img_list)
#   except:
#     count += 1
#     continue
result = pano_imgs(img_list)
# print(f"Retries: {count}")
result = cv2.cvtColor(result, cv2.COLOR_BGR2RGB)
plt.imshow(result)
plt.axis("off")
plt.show()

1. Corner Detection

We applied the Harris corner detection algorithm to find corners within an image. The image was first converted to greyscale before we applied the corner-detecting algorithm using cv2.cornerHarris. This is basically a method used in the computation of the gradient of the pixel intensity to then detect corners.

The most prominent problem encountered in this project involved the over-detection of corners where regions were highly textured or noisy. One Threshold is applied to select only the most relevant corners. It reduces irrelevant points of corners while keeping them distributed all over the image.

Detected corners are marked in red in the corner-detected image, highlighting important points in the image.

 2. Adaptive Non-Maximal Suppression (ANMS)

We implemented ANMS following the standard formulation: for each candidate corner, compute the radius to the nearest stronger corner and retain the N highest-radius points. Initially, too many redundant corners were selected in high-texture areas. Tuning the radius-based selection to balance strength and spatial separation reduced redundancy and produced a more uniform spatial distribution.

The resulting image after the application of ANMS contains fewer, yet well-scattered, corners marked in red. These are more likely to yield good matches for stitching.

 3. Feature Descriptors

We implemented a simple patch-based descriptor with minor modifications for edge cases.

Corners detected near the edge of the image can yield incomplete patches. Instead of zero-padding, we slightly shift the 40×40 window inward when a keypoint is too close to an edge. In practice this produced more stable descriptors than padding. Dealing with this edge case ensures we always have a 40×40 patch to work with, which prevents errors in descriptor computation.

The following images show examples of the 8×8 patches around corners (or shifted near boundaries). Each patch is normalized to improve robustness to lighting changes.

4. Feature Matching

We use nearest-neighbor matching under SSD distance with a ratio test to reject ambiguous correspondences. Initially, many regions produced similar descriptors, yielding false matches. Tightening the ratio threshold helped filter weak matches and improved robustness and accuracy.

The figures show matched points for the sample image pairs, with lines connecting corresponding features. Only confident matches are visualized after the ratio test.



 5. RANSAC

We implemented a RANSAC loop for robust homography estimation.

Choosing the number of iterations N and the inlier threshold t is a trade-off. Too few iterations yield poor estimates; too many slow the process. After experimentation, we used N = 1000 and an inlier threshold proportional to the number of matches (int(np.ceil(len(matches) * 0.9))). This dynamic thresholding improved stability of the estimated homographies.

Below we show inlier matches used to compute the homography, which maps points from one image to the other for alignment.

 6. Image Warping and Blending

This function takes two images and performs a reverse homography to transform img2 into the frame of img1. Corners are warped on img2 based on the homography. The bounding box is computed from the combination of all corners from img1 and warped img2 to define the panorama canvas. We then create masks to identify overlapping and non-overlapping regions and blend using seamlessClone.

A prior issue was excessive black regions after blending due to aggressive zero-padding of the canvas; adjusting the canvas computation and masks alleviated this.

Output image alignment demonstrates the accuracy of homography calculations. No varying brightness in different parts of the image, showing successful use of seamlessClone. The black regions around the image, especially upper-left, are a result of panorama space being larger than combined images.

7. Image Stitching/Panorama

We integrated the full pipeline to stitch sequences with more than two images. Our initial approach warped and blended image-by-image while recomputing features on intermediate mosaics, which degraded keypoint quality. We switched to estimating homographies between consecutive original images first, then composing transforms and applying warps/blending at the end. This improved stability on sequences with moderate viewpoint changes.

On sequences with stronger perspective change, distortion can cause parts of the final image to fall outside the canvas. A practical improvement is to project images to a cylindrical or spherical surface prior to matching/warping, which generally yields better alignment for wider fields of view.